# Vectorization, Memory Efficiency & Performance Optimization

## Definition
Writing fast NumPy code is about:
1. **Avoiding Python loops** → use vectorized operations
2. **Using right dtypes** → float32 vs float64 uses half the memory and is faster on GPU
3. **Understanding memory layout** → C vs Fortran order, contiguous access patterns
4. **Avoiding unnecessary copies** → use views when possible
5. **Using in-place operations** → `+=`, `*=` avoid allocating new arrays
6. **Profiling** → always measure before optimizing

### Tools for Profiling
- `%timeit` in Jupyter
- `time.time()` for coarse timing
- `numpy.testing` for correctness checks
- `memory_profiler` for memory usage

In [ ]:
import numpy as np
import time

# Rule 1: Avoid Python loops — use vectorized ops
N = 500_000
arr = np.random.randn(N)

# SLOW: Python loop
t0 = time.time()
result = [1.0 / (1.0 + np.exp(-x)) for x in arr]
print(f'Python loop sigmoid: {time.time()-t0:.4f}s')

# FAST: Vectorized
t0 = time.time()
result = 1.0 / (1.0 + np.exp(-arr))
print(f'NumPy vectorized:    {time.time()-t0:.4f}s')

In [ ]:
# Rule 2: Choose the right dtype (float32 vs float64)
N = 1000
A64 = np.random.randn(N, N).astype(np.float64)
A32 = A64.astype(np.float32)

t0 = time.time()
_ = A64 @ A64
print(f'float64 matmul: {time.time()-t0:.4f}s  memory={A64.nbytes//1024}KB')

t0 = time.time()
_ = A32 @ A32
print(f'float32 matmul: {time.time()-t0:.4f}s  memory={A32.nbytes//1024}KB')

In [ ]:
# Rule 3: Memory layout — contiguous access is faster
n = 2000
A = np.random.randn(n, n)

# Column-sum: axis=0 — accesses memory in strides (bad for C-order)
t0 = time.time()
_ = np.sum(A, axis=0)
t1 = time.time()
print(f'Sum axis=0 (strided): {t1-t0:.4f}s')

# Row-sum: axis=1 — accesses memory sequentially (good for C-order)
t0 = time.time()
_ = np.sum(A, axis=1)
t1 = time.time()
print(f'Sum axis=1 (seq):     {t1-t0:.4f}s')

# Make contiguous if needed
A_T_contiguous = np.ascontiguousarray(A.T)
t0 = time.time()
_ = np.sum(A_T_contiguous, axis=1)
print(f'Sum on contiguous T:  {time.time()-t0:.4f}s')

In [ ]:
# Rule 4: Use views instead of copies when safe
data = np.random.randn(10000, 100)

# BAD: creates a copy then normalizes
t0 = time.time()
batch = data[:32].copy()
batch = batch / batch.max()
print(f'With copy: {time.time()-t0:.6f}s')

# BETTER: normalize in-place (if you don't need original)
t0 = time.time()
batch2 = data[:32]
batch2 = batch2 / batch2.max()  # new array but no unnecessary .copy()
print(f'No copy:   {time.time()-t0:.6f}s')

In [ ]:
# Rule 5: In-place operations avoid extra allocation
arr = np.random.randn(1_000_000)

t0 = time.time()
result = arr + 1.0  # creates new array
print(f'arr + 1.0 (new alloc): {time.time()-t0:.4f}s')

arr2 = arr.copy()
t0 = time.time()
arr2 += 1.0  # in-place, no new allocation
print(f'arr += 1.0 (in-place): {time.time()-t0:.4f}s')

In [ ]:
# Rule 6: Pre-allocate output arrays (avoid repeated allocations in loops)
n_iter = 100
n = 1000

# BAD: allocate inside loop
t0 = time.time()
for _ in range(n_iter):
    result = np.zeros(n)
    result += np.random.randn(n)
print(f'Allocate in loop: {time.time()-t0:.4f}s')

# BETTER: pre-allocate and reuse
t0 = time.time()
result = np.zeros(n)
for _ in range(n_iter):
    result[:] = 0   # zero without new allocation
    result += np.random.randn(n)
print(f'Pre-allocated:    {time.time()-t0:.4f}s')

In [ ]:
# Rule 7: Use np.einsum for complex multi-dimensional ops
import numpy as np

# Batch matrix multiplication with einsum
A = np.random.randn(32, 10, 64)  # (batch, seq, features)
B = np.random.randn(32, 64, 10)  # (batch, features, seq)

t0 = time.time()
r1 = np.matmul(A, B)
print(f'matmul:  {time.time()-t0:.4f}s  shape={r1.shape}')

t0 = time.time()
r2 = np.einsum('bid,bdo->bio', A, B)
print(f'einsum:  {time.time()-t0:.4f}s  shape={r2.shape}')
print('Match:', np.allclose(r1, r2))

In [ ]:
# Rule 8: np.ascontiguousarray for C-order guarantee before intensive ops
mat = np.random.randn(1000, 1000)
mat_t = mat.T   # F-order view — may be slower for some operations

# Force C-order
mat_t_c = np.ascontiguousarray(mat_t)
print('mat.T is C-contiguous:', mat_t.flags['C_CONTIGUOUS'])
print('ascontiguousarray is C-contiguous:', mat_t_c.flags['C_CONTIGUOUS'])

In [ ]:
# Memory usage tracking
import sys

arrays = {
    'int8 (1M)':   np.ones(1_000_000, dtype=np.int8),
    'int32 (1M)':  np.ones(1_000_000, dtype=np.int32),
    'float16 (1M)': np.ones(1_000_000, dtype=np.float16),
    'float32 (1M)': np.ones(1_000_000, dtype=np.float32),
    'float64 (1M)': np.ones(1_000_000, dtype=np.float64),
}

print(f'{"Name":20s}  {"nbytes":>10s}  {"MB":>8s}')
for name, arr in arrays.items():
    print(f'{name:20s}  {arr.nbytes:>10,}  {arr.nbytes/1024/1024:>8.2f}')

In [ ]:
# Profiling tip: use %timeit in Jupyter (shows here as function call)
import timeit

def test_loop(n=10000):
    result = 0.0
    arr = list(range(n))
    for x in arr:
        result += x**2
    return result

def test_numpy(n=10000):
    arr = np.arange(n, dtype=np.float64)
    return np.sum(arr**2)

# Manual timing
loop_time = timeit.timeit(test_loop, number=100) / 100
numpy_time = timeit.timeit(test_numpy, number=100) / 100
print(f'Python loop: {loop_time*1000:.3f}ms')
print(f'NumPy:       {numpy_time*1000:.3f}ms')
print(f'Speedup:     {loop_time/numpy_time:.1f}x')